# 03 - Gold and Processed Delta Tables

Aggregate dashboard metrics into gold and publish the clean event-level silver data as a downstream processed Delta table.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)) if str(ROOT) not in sys.path else None

from insightsstream.delta_pipeline import DeltaSettings, build_gold_and_processed, create_delta_spark_session, delta_history

settings = DeltaSettings.from_env()
spark = create_delta_spark_session("InsightsStream-Gold-Delta", settings)

In [ ]:
gold_rows = build_gold_and_processed(spark, settings)
print(f"gold rows={gold_rows}")
print(f"processed Delta={settings.uri('processed/events')}")

gold = spark.read.format("delta").load(settings.uri("gold/daily_metrics"))
gold.createOrReplaceTempView("gold_daily_metrics")
spark.sql("SELECT * FROM gold_daily_metrics ORDER BY event_date, total_value DESC").show(100, truncate=False)
delta_history(spark, "processed/events", settings).select("version", "timestamp", "operation").show(truncate=False)